In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import(
accuracy_score,
f1_score,
precision_score,
recall_score,
confusion_matrix,
roc_auc_score)
from sklearn.model_selection import GridSearchCV ,cross_val_score,cross_val_predict
from sklearn.pipeline import Pipeline

Part 1:

In [ ]:
df = pd.read_csv('appointments_clean.csv')

In [ ]:
df.head()

In [ ]:
df['reminder_sent'] = df['reminder_sent'].map({'Yes': 1,'No':0})

In [ ]:
selected_feat = [
    'age',
    'distance_km',
    'fee_inr',
    'booking_channel',
    'visit_type',
    'reminder_sent',	
    'prior_no_shows',
    'lead_days'
]

In [ ]:
X_base = df[selected_feat].copy()
y = df['no_show']

In [ ]:
X = pd.get_dummies(X_base, columns=["booking_channel","visit_type"],dtype=float)


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size= 0.2,random_state = 42)

In [ ]:
print(f'training set size : {X_train.shape}')
print(f'test set size : {X_test.shape}')
print(f'No. of no-shows in test set :{y_test.sum()}')


In [ ]:
baseline_accuracy = (y_test == 0).mean()
print(f"Baseline Accuracy of always predicting 'Attended' :{baseline_accuracy:.3f}")

Part 2:

In [ ]:
X_test.dtypes

In [ ]:
scaler  = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test) 

In [ ]:
lr_model = LogisticRegression(max_iter = 1000,random_state =42)
lr_model.fit(X_train_s,y_train)

In [ ]:
y_pred_lr = lr_model.predict(X_test_s)
y_prob_lr = lr_model.predict_proba(X_test_s)[:,1]

In [ ]:
print("--- Logistic Regression Test Metrics ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.2f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.2f}")
print(f"Recall:    {recall_score(y_test, y_pred_lr):.2f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_lr):.2f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.2f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

In [ ]:
print("\n--- Feature Coefficients ---")
for col, coef in zip(X.columns, lr_model.coef_[0]):
  print(f"{col}: {coef:.4f}")

In [ ]:
reminder_idx = list(X.columns).index("reminder_sent")
raw_coef = lr_model.coef_[0][reminder_idx]
feature_std = scaler.scale_[reminder_idx]

In [ ]:
exp_coef = np.exp(raw_coef)
true_reminder_effect = np.exp(raw_coef / feature_std)

In [ ]:
print(
    f"\nRaw model coefficient for reminder_sent: {raw_coef:.2f} -> exp(coef):"
    f" {exp_coef:.2f}"
)
print(
    f"True effect of reminder_sent (divided by feature std {feature_std:.2f}):"
    f" {true_reminder_effect:.2f}"
)

Part 3:

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train,y_train)

In [ ]:
train_acc_full = accuracy_score(y_train, dt_model.predict(X_train))
test_acc_full = accuracy_score(y_test, dt_model.predict(X_test))

In [ ]:
print(
    f"Unbounded Tree -> Train Acc: {train_acc_full:.2f} | Test Acc:"
    f" {test_acc_full:.2f}"
)

In [ ]:
dt_d3_model =DecisionTreeClassifier(max_depth =3,random_state=42)
dt_d3_model.fit(X_train,y_train)

In [ ]:
train_acc_d3 = accuracy_score(y_train, dt_d3_model.predict(X_train))
test_acc_d3 = accuracy_score(y_test, dt_d3_model.predict(X_test))

In [ ]:
print(
    f"Depth-3 Tree   -> Train Acc: {train_acc_d3:.2f} | Test Acc:"
    f" {test_acc_d3:.2f}"
)

Part 4:

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100,random_state=42)
rf_model.fit(X_train,y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:,1]

In [ ]:
print(f'Random Forest Training Accuracy Score :{accuracy_score(y_train,rf_model.predict(X_train)):.2f}')
print(f'Random Forest Accuracy Score :{accuracy_score(y_test,y_pred_rf):.2f}')
print(f'Random Forest Recall Score :{recall_score(y_test,y_pred_rf):.2f}')
print(f'Random Forest ROC-AUC Score :{roc_auc_score(y_test,y_prob_rf):.2f}')

In [ ]:
importance = pd.Series(rf_model.feature_importances_, index = X.columns)
print('\n Top 3 Important Features :')
print(importance.nlargest(3))

In [ ]:
comparison_data = {
    'Model':['Logistic Regression','3 Depth Decision tree ','Random Forest Classifier'],
    'Accuracy':[accuracy_score(y_test, y_pred_lr),
                test_acc_d3,
                accuracy_score(y_test,rf_model.predict(X_test))
               ],
    'Recall' :[recall_score(y_test, y_pred_lr),
               recall_score(y_test, dt_d3_model.predict(X_test)),
               recall_score(y_test,y_pred_rf)
              ],
    'ROC-AUC':[roc_auc_score(y_test, y_prob_lr),
               roc_auc_score(y_test, dt_d3_model.predict_proba(X_test)[:,1]),
               roc_auc_score(y_test,y_prob_rf) 
              ]
}
comparison_df = pd.DataFrame(comparison_data)    
print('\n',comparison_df.to_string(index = False))   

Part 5:

In [ ]:
param_grid = {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc"
    )

grid_search.fit(X_train,y_train)

In [ ]:
print(f'Best Parameter : {grid_search.best_params_}')
print(f'Best ROC-AUC Score :{grid_search.best_score_:.2f}')

In [ ]:
lr_pipeline = Pipeline([
    ('scaler',StandardScaler()),
    ('clf',LogisticRegression(max_iter=1000))
])

In [ ]:
cv_lr = cross_val_score(lr_pipeline,
                        X_train, y_train,
                        cv=5,
                        scoring="roc_auc").mean()

In [ ]:
cv_dt_d3 = cross_val_score(DecisionTreeClassifier(max_depth =3,random_state=42),
                           X_train, y_train,
                           cv=5,
                           scoring="roc_auc").mean()

In [ ]:
cv_rf = cross_val_score(RandomForestClassifier(n_estimators=100,random_state=42),
                        X_train, y_train,
                        cv=5,
                        scoring="roc_auc").mean()

In [ ]:
cv_tuned_rf = grid_search.best_score_

In [ ]:
print("\n--- 5-Fold Cross-Validated ROC-AUC Comparison ---")
print(f"Logistic Regression: {cv_lr:.2f}")
print(f"Depth-3 Tree:        {cv_dt_d3:.2f}")
print(f"Standard RF:         {cv_rf:.2f}")
print(f"Tuned RF:            {cv_tuned_rf:.2f}")

In [ ]:
best_model = lr_pipeline
print(best_model)

In [ ]:
oof_train_probs = cross_val_predict(best_model, X_train, y_train,
                  cv=5,
                  method="predict_proba")[:, 1] 

In [ ]:
for thres in [0.5,0.35]:
    pred = (oof_train_probs >= thres).astype(int)
    print(f"\nThreshold: {thres}")
    print(f"  Precision: {precision_score(y_train, pred):.2f}")
    print(f"  Recall:    {recall_score(y_train, pred):.2f}")
    print(f"  F1 Score:  {f1_score(y_train, pred,):.2f}")
    print(f"  Flagged:   {pred.sum()} out of {len(y_train)}")    

In [ ]:
chosen_thres = 0.35

In [ ]:
appointment_id = df['appointment_id']

In [ ]:
X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, appointment_id,
    test_size=0.2,
    random_state=42
)

In [ ]:
best_model.fit(X_train,y_train)
final_test_prob = best_model.predict_proba(X_test)[:,1]
final_pred_test = (final_test_prob >= chosen_thres).astype(int)

In [ ]:
print("\n--- Final Test Evaluation (Once Only) ---")
print(f"Precision: {precision_score(y_test, final_pred_test):.2f}")
print(f"Recall:    {recall_score(y_test, final_pred_test):.2f}")
print(f"Flagged:   {final_pred_test.sum()} out of {len(X_test)}")
print(f"ROC-AUC:   {roc_auc_score(y_test, final_test_prob):.2f}")

In [ ]:
test_results_df = pd.DataFrame({
    "appointment_id": appointment_id.loc[X_test.index],
    "no_show": y_test.values,
    "risk_prob": final_test_prob,
    "flag": final_pred_test,
})
test_results_df.to_csv("test_risk_scores.csv", index=False)
print("\nSaved test_risk_scores.csv successfully.")